# 📊 GRIDBREAKERS — Pipeline Tổng Hợp
## Gộp 3 file phân tích:
- **File 1**: `cor_điều_chỉnh.py` — Correlation Analysis + VIF + Linear Regression
- **File 2**: `sarima.py` — SARIMA Pipeline
- **File 3**: `model.py` — Deep Learning & Ensemble Pipeline (XGBoost, RF, LSTM, TFT)

> Chạy từng cell theo thứ tự. Upload các file CSV lên Colab trước khi chạy.

## ⚙️ 0. Cài đặt thư viện

In [ ]:
# Cài đặt các thư viện cần thiết (chạy 1 lần)
!pip install xgboost shap torch statsmodels scikit-learn pandas numpy matplotlib seaborn -q


## 📁 Upload dữ liệu CSV

In [ ]:
from google.colab import files
import os

print('Upload các file CSV: sales.csv, orders.csv, order_items.csv, payments.csv, shipments.csv, returns.csv, web_traffic.csv')
uploaded = files.upload()
print('\n✅ Đã upload:', list(uploaded.keys()))


---
# 📈 FILE 1 — Correlation Analysis + VIF + Linear Regression


### 1.1 Import & Đọc Dữ Liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. ĐỌC DỮ LIỆU
# ============================================================
print("Đang đọc dữ liệu...")

sales      = pd.read_csv('sales.csv',       parse_dates=['Date'])
orders     = pd.read_csv('orders.csv',      parse_dates=['order_date'])
order_items= pd.read_csv('order_items.csv')
payments   = pd.read_csv('payments.csv')
shipments  = pd.read_csv('shipments.csv',   parse_dates=['ship_date', 'delivery_date'])
returns    = pd.read_csv('returns.csv',     parse_dates=['return_date'])
web        = pd.read_csv('web_traffic.csv', parse_dates=['date'])

print("✅ Đọc xong!")


### 1.2 Xử Lý Bảng & Merge

In [ ]:

# ============================================================
# 2. XỬ LÝ TỪNG BẢNG
# ============================================================

# ----------------------------------------------------------
# 2.1 SALES — Nguồn revenue chính
# ----------------------------------------------------------
sales = sales.rename(columns={'Date': 'date', 'Revenue': 'revenue', 'COGS': 'cogs'})
sales['date'] = sales['date'].dt.normalize()

# ----------------------------------------------------------
# 2.2 SHIPMENTS — Tính delivery_days = delivery_date - ship_date
# ----------------------------------------------------------
# delivery_days là thuộc tính của từng đơn hàng
# → merge vào orders theo order_id, sau đó aggregate theo order_date
shipments['delivery_days'] = (
    shipments['delivery_date'] - shipments['ship_date']
).dt.days

# Chỉ lấy 2 cột cần thiết
shipments_slim = shipments[['order_id', 'shipping_fee', 'delivery_days']]

# ----------------------------------------------------------
# 2.3 ORDER ITEMS — Aggregate theo order_id
# discount_amount (sum), quantity (sum), has_promotion (flag)
# ----------------------------------------------------------
order_items_agg = order_items.groupby('order_id', as_index=False).agg(
    discount_amount = ('discount_amount', 'sum'),
    # Flag = 1 nếu đơn hàng có ít nhất 1 dòng có promo_id không null
    has_promotion = ('promo_id', lambda x: x.notna().sum())
)


# ----------------------------------------------------------
# 2.4 PAYMENTS — Đếm số đơn thanh toán bằng credit_card
# Giữ nguyên cấp order_id, aggregate theo ngày ở bước sau
# ----------------------------------------------------------
payments_slim = payments[['order_id', 'payment_method']]

# ----------------------------------------------------------
# 2.5 BUILD ORDER-LEVEL TABLE
# Gắn tất cả thông tin vào bảng orders (cấp order_id)
# ----------------------------------------------------------
orders_full = (
    orders[['order_id', 'order_date']]
    .merge(shipments_slim,   on='order_id', how='left')
    .merge(order_items_agg,  on='order_id', how='left')
    .merge(payments_slim,    on='order_id', how='left')
)

# Fill NaN cho các cột numeric
for col in ['shipping_fee', 'delivery_days', 'discount_amount',
     'has_promotion']:
    orders_full[col] = orders_full[col].fillna(0)

orders_full['order_date'] = orders_full['order_date'].dt.normalize()

# ----------------------------------------------------------
# 2.6 AGGREGATE ORDERS THEO NGÀY
# ----------------------------------------------------------
daily_orders = orders_full.groupby('order_date', as_index=False).agg(
    shipping_fee      = ('shipping_fee',   'sum'),
    discount_amount   = ('discount_amount','sum'),
    # Tổng số đơn có khuyến mãi trong ngày
    promo_orders      = ('has_promotion',  'sum'),
    # Đếm số đơn thanh toán bằng credit_card
    credit_card       = ('payment_method', lambda x: (x == 'credit_card').sum())
).rename(columns={'order_date': 'date'})
# ----------------------------------------------------------
# 2.7 RETURNS — Aggregate theo return_date (ngày thực tế trả)
# ----------------------------------------------------------
daily_returns = returns.groupby('return_date', as_index=False).agg(
    num_returns_orders = ('return_id',       'count'),  # count by day
    refund_amount      = ('refund_amount',   'sum'),
    num_return_qty     = ('return_quantity', 'sum')
).rename(columns={'return_date': 'date'})
daily_returns['date'] = daily_returns['date'].dt.normalize()

# ----------------------------------------------------------
# 2.8 WEB TRAFFIC — Aggregate theo date
# SUM cho count metrics, giữ nguyên (first) cho avg_session_duration_sec
# vì đây là giá trị đã được tính sẵn ở cấp ngày trong bảng gốc
# ----------------------------------------------------------

# Kiểm tra xem web traffic đã ở cấp ngày chưa
# (nếu mỗi ngày chỉ có 1 dòng thì không cần aggregate)
rows_per_date = web.groupby('date')['sessions'].count()
has_multiple_rows = (rows_per_date > 1).any()

# Mỗi ngày chỉ có 1 dòng → giữ nguyên, không cần aggregate
daily_web = web[['date', 'sessions', 'unique_visitors', 'avg_session_duration_sec']].copy()

daily_web['date'] = daily_web['date'].dt.normalize()


# ============================================================
# 3. MERGE TẤT CẢ VÀO BẢNG DAILY
# ============================================================
print("Đang merge...")

df = sales.copy()

merge_tables = [
    (daily_orders,  'date'),
    (daily_returns, 'date'),
    (daily_web,     'date')
]

for table, key in merge_tables:
    df = df.merge(table, on=key, how='left')


# Fill 0 cho các cột count/sum còn lại
fill_zero_cols = [
    'shipping_fee', 'delivery_days', 'discount_amount', 
    'promo_orders', 'credit_card',
    'num_returns_orders', 'refund_amount', 'num_return_qty',
    'total_sessions', 'total_unique_visitors'
]
for col in fill_zero_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# ----------------------------------------------------------
# 3.1 FEATURE ENGINEERING
# ----------------------------------------------------------
df['date']           = pd.to_datetime(df['date'])
df['month']          = df['date'].dt.month
df['day_of_week']    = df['date'].dt.dayofweek       # 0=Monday
df['is_peak_season'] = df['month'].isin([4, 5, 6]).astype(int)

print(f"✅ Final shape: {df.shape}")
print(f"   Date range: {df['date'].min().date()} → {df['date'].max().date()}")



### 1.3 Correlation Analysis & Visualization

In [ ]:
# ============================================================
# 4. CORRELATION ANALYSIS
# ============================================================
print("\nĐang tính correlation...")

feature_cols = [
    'shipping_fee', 'delivery_days', 'discount_amount',
     'promo_orders', 'credit_card',
    'num_returns_orders', 'refund_amount', 'num_return_qty',
    'total_sessions', 'total_unique_visitors',
    'avg_session_duration_sec',
    'is_peak_season'
]

# Chỉ giữ cột thực sự tồn tại
feature_cols = [c for c in feature_cols if c in df.columns]
all_cols     = ['revenue'] + feature_cols

# Tính Pearson correlation + p-value
corr_results = []
for col in feature_cols:
    valid = df[['revenue', col]].dropna()
    if len(valid) > 1:
        r, p = pearsonr(valid['revenue'], valid[col])
        corr_results.append({
            'feature':     col,
            'correlation': r,
            'p_value':     p,
            'significant': '✅' if p < 0.05 else '❌'
        })

corr_df = (
    pd.DataFrame(corr_results)
    .sort_values('correlation', ascending=False)
    .reset_index(drop=True)
)

print("\n" + "="*65)
print(f"{'Feature':<28} {'Corr':>8} {'p-value':>12} {'Sig?':>6}")
print("="*65)
for _, row in corr_df.iterrows():
    print(f"{row['feature']:<28} {row['correlation']:>8.4f} "
          f"{row['p_value']:>12.4e} {row['significant']:>6}")

# ============================================================
# 5. VISUALISATION
# ============================================================

# --- Màu sắc theo chiều tương quan ---
colors = [
    '#D32F2F' if v > 0 else '#1976D2'
    for v in corr_df['correlation']
]
# Làm mờ nếu không significant
alphas = [
    1.0 if p < 0.05 else 0.35
    for p in corr_df['p_value']
]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')

# ── Plot 1: Bar chart correlation với revenue ──────────────
ax1 = axes[0]
bars = ax1.barh(
    corr_df['feature'],
    corr_df['correlation'],
    color=colors,
    alpha=0.9,
    edgecolor='white',
    linewidth=0.4,
    height=0.65
)

# Làm mờ bar không significant
for bar, alpha in zip(bars, alphas):
    bar.set_alpha(alpha)

# Vẽ đường 0
ax1.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)

# Thêm giá trị và dấu significance lên bar
for i, (_, row) in enumerate(corr_df.iterrows()):
    val  = row['correlation']
    sig  = row['significant']
    xpos = val + (0.01 if val >= 0 else -0.01)
    ha   = 'left' if val >= 0 else 'right'
    ax1.text(xpos, i, f"{val:.3f} {sig}",
             va='center', ha=ha, fontsize=8.5,
             color='white', fontweight='bold')

ax1.set_xlabel('Pearson Correlation Coefficient', color='white', fontsize=11)
ax1.set_title('Correlation with Revenue\n(Red = Positive | Blue = Negative | Faded = p≥0.05)',
              color='white', fontsize=12, fontweight='bold', pad=12)
ax1.tick_params(colors='white', labelsize=9)
ax1.spines['bottom'].set_color('#444')
ax1.spines['left'].set_color('#444')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_xlim(-1.05, 1.05)
ax1.invert_yaxis()

# ── Plot 2: Full Correlation Heatmap ──────────────────────
ax2 = axes[1]
corr_matrix = df[all_cols].corr(method='pearson')

mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # Ẩn tam giác trên

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.3,
    linecolor='#1a1a2e',
    annot_kws={'size': 7, 'weight': 'bold'},
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
    ax=ax2
)

ax2.set_title('Full Correlation Matrix\n(Lower Triangle — Pearson)',
              color='white', fontsize=12, fontweight='bold', pad=12)
ax2.tick_params(colors='white', labelsize=8)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)

# Colorbar label màu trắng
cbar = ax2.collections[0].colorbar
cbar.ax.yaxis.label.set_color('white')
cbar.ax.tick_params(colors='white')

plt.suptitle(
    'GRIDBREAKERS — Feature Correlation Analysis with Daily Revenue',
    color='#39ff14', fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('correlation_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("\n💾 Đã lưu: correlation_analysis.png")


### 1.4 VIF Analysis

In [ ]:


#VIF
# ============================================================
# 1. CHỌN BIẾN |r| > 0.3
# ============================================================
selected_features = [
    #'credit_card',
    'shipping_fee',
    'is_peak_season',
    #'num_returns_orders',
    #'num_return_qty',
    'refund_amount',
    'fill_rate',
    'days_of_supply'
]

# Chỉ giữ các cột tồn tại trong df
selected_features = [c for c in selected_features if c in df.columns]

# ============================================================
# 2. CHUẨN BỊ DATA
# Drop NaN vì VIF không chấp nhận missing values
# ============================================================
df_vif = df[selected_features].dropna().reset_index(drop=True)

print(f"Shape sau khi drop NaN: {df_vif.shape}")
print(f"Số dòng bị drop: {len(df) - len(df_vif)}")

# ============================================================
# 3. TÍNH VIF
# VIF = 1/(1 - R²) của từng biến khi regress lên các biến còn lại
# Không thêm constant vì VIF đã xử lý nội tại
# ============================================================
def calculate_vif(data):
    """
    Tính VIF cho tất cả các biến trong dataframe.
    Thêm constant để tính đúng theo hồi quy tuyến tính.
    """
    data_with_const = add_constant(data)
    
    vif_data = []
    for i, col in enumerate(data.columns):
        # i+1 vì cột 0 là constant
        vif_val = variance_inflation_factor(
            data_with_const.values, i + 1
        )
        vif_data.append({
            'feature': col,
            'VIF':     round(vif_val, 4)
        })
    
    return (
        pd.DataFrame(vif_data)
        .sort_values('VIF', ascending=False)
        .reset_index(drop=True)
    )

vif_df = calculate_vif(df_vif)

# ============================================================
# 4. GÁN NHÃN MỨC ĐỘ MULTICOLLINEARITY
# ============================================================
def vif_label(vif):
    if vif >= 10:
        return '🔴 Severe   (loại bỏ)'
    elif vif >= 5:
        return '🟡 Moderate (cân nhắc)'
    elif vif >= 2.5:
        return '🟢 Low      (chấp nhận)'
    else:
        return '✅ Minimal  (tốt)'

vif_df['level']     = vif_df['VIF'].apply(vif_label)
vif_df['bar_color'] = vif_df['VIF'].apply(
    lambda v: '#D32F2F' if v >= 10
    else '#FF9800' if v >= 5
    else '#4CAF50' if v >= 2.5
    else '#1976D2'
)

# In bảng kết quả
print("\n" + "=" * 60)
print(f"{'Feature':<22} {'VIF':>8}   {'Mức độ'}")
print("=" * 60)
for _, row in vif_df.iterrows():
    print(f"{row['feature']:<22} {row['VIF']:>8.4f}   {row['level']}")

# ============================================================
# 5. ITERATIVE VIF — Tự động loại biến VIF >= 10
# ============================================================
print("\n" + "=" * 60)
print("ITERATIVE VIF — Loại dần biến có VIF cao nhất")
print("=" * 60)

def iterative_vif(data, threshold=10):
    """
    Lặp lại: tính VIF → loại biến có VIF cao nhất vượt threshold
    → tính lại đến khi tất cả VIF < threshold
    """
    remaining = data.copy()
    removed   = []
    iteration = 1

    while True:
        vif_result = calculate_vif(remaining)
        max_vif    = vif_result['VIF'].max()
        max_feat   = vif_result.loc[vif_result['VIF'].idxmax(), 'feature']

        print(f"\nIteration {iteration}:")
        print(vif_result.to_string(index=False))

        if max_vif < threshold:
            print(f"\n✅ Tất cả VIF < {threshold} — Dừng lại.")
            break

        print(f"\n⚠️  Loại '{max_feat}' (VIF = {max_vif:.4f})")
        removed.append({'feature': max_feat, 'VIF_khi_loai': round(max_vif, 4)})
        remaining = remaining.drop(columns=[max_feat])
        iteration += 1

    return remaining.columns.tolist(), removed

final_features, removed_features = iterative_vif(df_vif, threshold=10)

print("\n" + "=" * 60)
print("KẾT QUẢ SAU ITERATIVE VIF")
print("=" * 60)
print(f"✅ Giữ lại : {final_features}")
if removed_features:
    print(f"❌ Loại bỏ : {[r['feature'] for r in removed_features]}")
    print("\nChi tiết biến bị loại:")
    for r in removed_features:
        print(f"   {r['feature']:<22} VIF = {r['VIF_khi_loai']}")

# ============================================================
# 6. VISUALISATION
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')

# ── Plot 1: VIF bar chart (tất cả biến ban đầu) ───────────
ax1 = axes[0]

bars = ax1.barh(
    vif_df['feature'],
    vif_df['VIF'],
    color=vif_df['bar_color'],
    edgecolor='white',
    linewidth=0.4,
    height=0.6
)

# Đường ngưỡng VIF = 5 và 10
ax1.axvline(5,  color='#FF9800', linewidth=1.5,
            linestyle='--', alpha=0.8, label='VIF = 5 (Moderate)')
ax1.axvline(10, color='#D32F2F', linewidth=1.5,
            linestyle='--', alpha=0.8, label='VIF = 10 (Severe)')

# Label giá trị
for i, (_, row) in enumerate(vif_df.iterrows()):
    ax1.text(row['VIF'] + 0.2, i,
             f"{row['VIF']:.2f}",
             va='center', ha='left',
             fontsize=9, color='white', fontweight='bold')

ax1.set_xlabel('VIF Score', color='white', fontsize=11)
ax1.set_title(
    'VIF — Tất cả biến được chọn\n'
    '(Red ≥ 10 | Orange ≥ 5 | Green ≥ 2.5 | Blue < 2.5)',
    color='white', fontsize=11, fontweight='bold', pad=10
)
ax1.tick_params(colors='white', labelsize=9)
ax1.legend(fontsize=8, facecolor='#16213e',
           labelcolor='white', loc='lower right')
for spine in ['top', 'right']:
    ax1.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax1.spines[spine].set_color('#444')
ax1.invert_yaxis()

# ── Plot 2: Correlation heatmap của các biến được chọn ────
ax2 = axes[1]

corr_selected = df_vif.corr(method='pearson')

mask = np.zeros_like(corr_selected, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

sns.heatmap(
    corr_selected,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.4,
    linecolor='#1a1a2e',
    annot_kws={'size': 8, 'weight': 'bold'},
    cbar_kws={'shrink': 0.8, 'label': 'Pearson r'},
    ax=ax2
)

ax2.set_title(
    'Correlation Matrix — Các biến được chọn\n'
    '(Phát hiện multicollinearity giữa features)',
    color='white', fontsize=11, fontweight='bold', pad=10
)
ax2.tick_params(colors='white', labelsize=8)
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0)

cbar = ax2.collections[0].colorbar
cbar.ax.yaxis.label.set_color('white')
cbar.ax.tick_params(colors='white')

plt.suptitle(
    'GRIDBREAKERS — VIF Analysis & Feature Multicollinearity',
    color='#39ff14', fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('vif_analysis.png', dpi=150,
            bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("\n💾 Đã lưu: vif_analysis.png")


### 1.5 So Sánh R² Hiệu Chỉnh (Model A vs Model B)

In [ ]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# HÀM TÍNH R² HIỆU CHỈNH
# Công thức: R²_adj = 1 - (1 - R²) × (n-1) / (n-k-1)
# n = số quan sát, k = số biến độc lập
# ============================================================
def adjusted_r2(r2, n, k):
    """
    r2 : R² của mô hình
    n  : số quan sát
    k  : số biến độc lập (không tính intercept)
    """
    return 1 - (1 - r2) * (n - 1) / (n - k - 1)


# ============================================================
# HAI BỘ BIẾN CẦN SO SÁNH
# ============================================================
model_specs = {
    'Model A — num_returns_orders': [
        'shipping_fee',
        'is_peak_season',
        'num_returns_orders'
    ],
    'Model B — num_return_qty': [
        'shipping_fee',
        'is_peak_season',
    
    ]
}

# ============================================================
# TÍNH R² VÀ R² HIỆU CHỈNH CHO TỪNG MÔ HÌNH
# ============================================================
results = []

for model_name, features in model_specs.items():

    # --- Kiểm tra cột tồn tại ---
    missing = [f for f in features if f not in df.columns]
    if missing:
        print(f"⚠️  {model_name}: thiếu cột {missing}")
        continue

    # --- Chuẩn bị data: drop NaN ---
    cols_needed = ['revenue'] + features
    data        = df[cols_needed].dropna().reset_index(drop=True)

    X = data[features].values
    y = data['revenue'].values
    n = len(y)
    k = len(features)

    # --- Fit Linear Regression ---
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)

    # --- Tính các chỉ số ---
    r2     = r2_score(y, y_pred)
    r2_adj = adjusted_r2(r2, n, k)
    ss_res = np.sum((y - y_pred) ** 2)          # Residual Sum of Squares
    ss_tot = np.sum((y - np.mean(y)) ** 2)      # Total Sum of Squares
    rmse   = np.sqrt(ss_res / n)
    mae    = np.mean(np.abs(y - y_pred))

    results.append({
        'model':       model_name,
        'features':    features,
        'n':           n,
        'k':           k,
        'R²':          round(r2,     6),
        'R²_adj':      round(r2_adj, 6),
        'RMSE':        round(rmse,   2),
        'MAE':         round(mae,    2),
        'intercept':   round(model.intercept_, 4),
        'coefs':       dict(zip(features, model.coef_.round(4))),
        'y_pred':      y_pred,
        'y_true':      y
    })

# ============================================================
# IN KẾT QUẢ
# ============================================================
print("\n" + "=" * 70)
print("SO SÁNH HAI MÔ HÌNH HỒI QUY TUYẾN TÍNH")
print("=" * 70)

for res in results:
    print(f"\n📌 {res['model']}")
    print(f"   Features  : {res['features']}")
    print(f"   n obs     : {res['n']:,}")
    print(f"   R²        : {res['R²']:.6f}")
    print(f"   R²_adj    : {res['R²_adj']:.6f}  ← chỉ số so sánh chính")
    print(f"   RMSE      : {res['RMSE']:,.2f}")
    print(f"   MAE       : {res['MAE']:,.2f}")
    print(f"   Intercept : {res['intercept']:,.4f}")
    print(f"   Hệ số     :")
    for feat, coef in res['coefs'].items():
        print(f"      {feat:<25}: {coef:,.4f}")

# So sánh trực tiếp
print("\n" + "=" * 70)
print("KẾT LUẬN SO SÁNH")
print("=" * 70)
if len(results) == 2:
    r2_adj_A = results[0]['R²_adj']
    r2_adj_B = results[1]['R²_adj']
    diff     = abs(r2_adj_A - r2_adj_B)
    winner   = results[0]['model'] if r2_adj_A > r2_adj_B else results[1]['model']
    loser    = results[1]['model'] if r2_adj_A > r2_adj_B else results[0]['model']

    print(f"   {results[0]['model']:<40}: R²_adj = {r2_adj_A:.6f}")
    print(f"   {results[1]['model']:<40}: R²_adj = {r2_adj_B:.6f}")
    print(f"   Chênh lệch R²_adj : {diff:.6f}")
    print(f"\n   ✅ Winner: {winner}")
    print(f"   ❌ Loser : {loser}")

    if diff < 0.001:
        print("\n   ⚠️  Chênh lệch < 0.001 → Hai mô hình tương đương nhau.")
        print("       Chọn biến có ý nghĩa kinh doanh rõ hơn.")

# ============================================================
# VISUALISATION — 3 BIỂU ĐỒ
# ============================================================
fig = plt.figure(figsize=(22, 14))
fig.patch.set_facecolor('#1a1a2e')

# Layout: 2 hàng × 3 cột
# Hàng 1: Actual vs Predicted cho từng model
# Hàng 2: Bar so sánh metrics, Residual plot

# ── Màu cho 2 model ──
MODEL_COLORS = ['#39ff14', '#FF9800']

# ----------------------------------------------------------
# Plot 1 & 2: Actual vs Predicted
# ----------------------------------------------------------
for idx, res in enumerate(results):
    ax = fig.add_subplot(2, 3, idx + 1)
    ax.set_facecolor('#16213e')

    y_true = res['y_true']
    y_pred = res['y_pred']

    # Scatter
    ax.scatter(y_true, y_pred,
               alpha=0.25, s=8,
               color=MODEL_COLORS[idx],
               label='Predicted vs Actual')

    # Perfect fit line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val],
            'w--', linewidth=1.2, alpha=0.7, label='Perfect fit')

    ax.set_xlabel('Actual Revenue',    color='white', fontsize=9)
    ax.set_ylabel('Predicted Revenue', color='white', fontsize=9)
    ax.set_title(
        f"{res['model']}\nR²={res['R²']:.4f} | R²_adj={res['R²_adj']:.4f}",
        color='white', fontsize=10, fontweight='bold', pad=8
    )
    ax.tick_params(colors='white', labelsize=8)
    ax.legend(fontsize=7, facecolor='#16213e', labelcolor='white')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['bottom', 'left']:
        ax.spines[spine].set_color('#444')

# ----------------------------------------------------------
# Plot 3: Bar chart so sánh R², R²_adj, RMSE (normalized)
# ----------------------------------------------------------
ax3 = fig.add_subplot(2, 3, 3)
ax3.set_facecolor('#16213e')

metrics    = ['R²', 'R²_adj']
x          = np.arange(len(metrics))
bar_width  = 0.3

for idx, res in enumerate(results):
    vals = [res['R²'], res['R²_adj']]
    bars = ax3.bar(x + idx * bar_width, vals,
                   width=bar_width,
                   color=MODEL_COLORS[idx],
                   alpha=0.85,
                   label=res['model'].split('—')[1].strip(),
                   edgecolor='white', linewidth=0.4)
    for bar, val in zip(bars, vals):
        ax3.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.002,
                 f'{val:.4f}',
                 ha='center', va='bottom',
                 fontsize=8, color='white', fontweight='bold')

ax3.set_xticks(x + bar_width / 2)
ax3.set_xticklabels(metrics, color='white', fontsize=10)
ax3.set_ylabel('Score', color='white', fontsize=10)
ax3.set_title('So sánh R² và R²_adj\ngiữa hai mô hình',
              color='white', fontsize=11, fontweight='bold', pad=8)
ax3.tick_params(colors='white', labelsize=9)
ax3.legend(fontsize=8, facecolor='#16213e', labelcolor='white')
ax3.set_ylim(0, max(results[0]['R²'], results[1]['R²']) * 1.15)
for spine in ['top', 'right']:
    ax3.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax3.spines[spine].set_color('#444')

# ----------------------------------------------------------
# Plot 4 & 5: Residual plot (y_true - y_pred)
# ----------------------------------------------------------
for idx, res in enumerate(results):
    ax = fig.add_subplot(2, 3, idx + 4)
    ax.set_facecolor('#16213e')

    residuals = res['y_true'] - res['y_pred']

    ax.scatter(res['y_pred'], residuals,
               alpha=0.25, s=8, color=MODEL_COLORS[idx])
    ax.axhline(0, color='white', linewidth=1.2,
               linestyle='--', alpha=0.7)

    ax.set_xlabel('Predicted Revenue', color='white', fontsize=9)
    ax.set_ylabel('Residual',          color='white', fontsize=9)
    ax.set_title(
        f"Residual Plot — {res['model'].split('—')[1].strip()}\n"
        f"RMSE={res['RMSE']:,.0f} | MAE={res['MAE']:,.0f}",
        color='white', fontsize=10, fontweight='bold', pad=8
    )
    ax.tick_params(colors='white', labelsize=8)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['bottom', 'left']:
        ax.spines[spine].set_color('#444')

# ----------------------------------------------------------
# Plot 6: RMSE & MAE so sánh
# ----------------------------------------------------------
ax6 = fig.add_subplot(2, 3, 6)
ax6.set_facecolor('#16213e')

error_metrics = ['RMSE', 'MAE']
x2 = np.arange(len(error_metrics))

for idx, res in enumerate(results):
    vals = [res['RMSE'], res['MAE']]
    bars = ax6.bar(x2 + idx * bar_width, vals,
                   width=bar_width,
                   color=MODEL_COLORS[idx],
                   alpha=0.85,
                   label=res['model'].split('—')[1].strip(),
                   edgecolor='white', linewidth=0.4)
    for bar, val in zip(bars, vals):
        ax6.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() * 1.01,
                 f'{val:,.0f}',
                 ha='center', va='bottom',
                 fontsize=8, color='white', fontweight='bold')

ax6.set_xticks(x2 + bar_width / 2)
ax6.set_xticklabels(error_metrics, color='white', fontsize=10)
ax6.set_ylabel('Error (VNĐ)', color='white', fontsize=10)
ax6.set_title('So sánh RMSE & MAE\n(thấp hơn = tốt hơn)',
              color='white', fontsize=11, fontweight='bold', pad=8)
ax6.tick_params(colors='white', labelsize=9)
ax6.legend(fontsize=8, facecolor='#16213e', labelcolor='white')
for spine in ['top', 'right']:
    ax6.spines[spine].set_visible(False)
for spine in ['bottom', 'left']:
    ax6.spines[spine].set_color('#444')

plt.suptitle(
    'GRIDBREAKERS — So sánh Model A vs Model B\n'
    'Linear Regression: R², R²_adj, Residuals',
    color='#39ff14', fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("\n💾 Đã lưu: model_comparison.png")

---
# 📉 FILE 2 — SARIMA Pipeline


### 2.1 Import & ETL

In [ ]:
"""
====================================================================
FILE 1 — SARIMA PIPELINE
====================================================================
3 biến ngoại sinh: shipping_fee, num_returns_orders, is_peak_season
Các bước:
  1.  ETL & Merge dữ liệu
  2.  Kiểm tra tính dừng (ADF + KPSS) → chọn d
  3.  Nhận diện p, q bằng ACF / PACF
  4.  Lựa chọn mô hình bằng AIC (grid-search nhỏ)
  5.  Huấn luyện SARIMAX (có biến ngoại sinh)
  6.  Chuẩn đoán phần dư (Ljung-Box, Q-Q, histogram)
  7.  Dự báo tập kiểm tra & so sánh với Ridge baseline
  8.  Visualization tổng hợp
====================================================================
"""

import warnings, itertools
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch

from statsmodels.tsa.stattools   import adfuller, kpss, acf, pacf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import scipy.stats as stats

from sklearn.linear_model  import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import r2_score, mean_absolute_error, mean_squared_error

# ─── Màu sắc ──────────────────────────────────────────────────────
C_BG    = '#0d0d1a'
C_PANEL = '#13132b'
C_NEON  = '#00e5ff'
C_GOLD  = '#ffd700'
C_RED   = '#ff4560'
C_GREEN = '#00e096'
C_GRAY  = '#555577'

plt.rcParams.update({
    'figure.facecolor': C_BG, 'axes.facecolor': C_PANEL,
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#333355', 'grid.color': '#1e1e3f',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
})

def style(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(C_PANEL)
    for s in ['top', 'right']:
        ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']:
        ax.spines[s].set_color('#333355')
    if title:  ax.set_title(title,  color=C_NEON, fontsize=10, fontweight='bold', pad=8)
    if xlabel: ax.set_xlabel(xlabel, color='#aaaacc', fontsize=8)
    if ylabel: ax.set_ylabel(ylabel, color='#aaaacc', fontsize=8)
    ax.grid(True)

# ====================================================================
# BƯỚC 1 — ETL & MERGE
# ====================================================================
print("=" * 70)
print("BƯỚC 1 — ETL & MERGE")
print("=" * 70)

sales       = pd.read_csv('sales.csv',       parse_dates=['Date'])
orders      = pd.read_csv('orders.csv',      parse_dates=['order_date'])
order_items = pd.read_csv('order_items.csv')
payments    = pd.read_csv('payments.csv')
shipments   = pd.read_csv('shipments.csv',   parse_dates=['ship_date','delivery_date'])
returns     = pd.read_csv('returns.csv',     parse_dates=['return_date'])

sales = sales.rename(columns={'Date':'date','Revenue':'revenue','COGS':'cogs'})
sales['date'] = sales['date'].dt.normalize()

shipments['delivery_days'] = (shipments['delivery_date'] - shipments['ship_date']).dt.days
shipments_slim = shipments[['order_id','shipping_fee','delivery_days']]

order_items_agg = order_items.groupby('order_id', as_index=False).agg(
    discount_amount=('discount_amount','sum'),
    quantity=('quantity','sum'),
    has_promotion=('promo_id', lambda x: int(x.notna().any()))
)
payments_slim = payments[['order_id','payment_method']]

orders_full = (
    orders[['order_id','order_date']]
    .merge(shipments_slim,  on='order_id', how='left')
    .merge(order_items_agg, on='order_id', how='left')
    .merge(payments_slim,   on='order_id', how='left')
)
for col in ['shipping_fee','delivery_days','discount_amount','quantity','has_promotion']:
    orders_full[col] = orders_full[col].fillna(0)
orders_full['order_date'] = orders_full['order_date'].dt.normalize()

daily_orders = orders_full.groupby('order_date', as_index=False).agg(
    shipping_fee    =('shipping_fee',   'sum'),
    discount_amount =('discount_amount','sum'),
    promo_orders    =('has_promotion',  'sum'),
    credit_card     =('payment_method', lambda x: (x=='credit_card').sum())
).rename(columns={'order_date':'date'})

daily_returns = returns.groupby('return_date', as_index=False).agg(
    num_returns_orders=('return_id','count'),
    refund_amount=('refund_amount','sum'),
    return_qty=('return_quantity','sum')
).rename(columns={'return_date':'date'})
daily_returns['date'] = daily_returns['date'].dt.normalize()

df = sales.copy()
for tbl in [daily_orders, daily_returns]:
    df = df.merge(tbl, on='date', how='left')

for col in ['shipping_fee','discount_amount','promo_orders','credit_card',
            'num_returns_orders','refund_amount','return_qty']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['is_peak_season'] = df['month'].isin([4,5,6]).astype(int)

# Tập train: 2012-07-04 → 2022-12-31
TRAIN_END = '2022-12-31'
TEST_START = '2023-01-01'
TEST_END   = '2024-07-01'

df_train = df[df['date'] <= TRAIN_END].copy().sort_values('date').reset_index(drop=True)
df_test  = df[(df['date'] >= TEST_START) & (df['date'] <= TEST_END)].copy().sort_values('date').reset_index(drop=True)

EXOG_COLS = ['shipping_fee', 'num_returns_orders', 'is_peak_season']
EXOG_COLS = [c for c in EXOG_COLS if c in df.columns]

y_train = df_train['revenue'].values
y_test  = df_test['revenue'].values if len(df_test) > 0 else None

exog_train = df_train[EXOG_COLS].values
exog_test  = df_test[EXOG_COLS].values if len(df_test) > 0 else None

print(f"✅ Train: {df_train['date'].min().date()} → {df_train['date'].max().date()} ({len(df_train):,} ngày)")
print(f"   Exogenous: {EXOG_COLS}")
if y_test is not None:
    print(f"   Test : {df_test['date'].min().date()} → {df_test['date'].max().date()} ({len(df_test):,} ngày)")



### 2.2 Kiểm Tra Tính Dừng (ADF + KPSS)

In [ ]:
# ====================================================================
# BƯỚC 2 — KIỂM TRA TÍNH DỪNG → CHỌN d
# ====================================================================
print("\n" + "=" * 70)
print("BƯỚC 2 — KIỂM TRA TÍNH DỪNG (ADF + KPSS)")
print("=" * 70)

def test_stationarity(series, name='Series', alpha=0.05):
    """ADF (H0: non-stationary) + KPSS (H0: stationary)"""
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series, autolag='AIC')
    kpss_stat, kpss_p, _, kpss_crit    = kpss(series, regression='c', nlags='auto')

    adf_ok  = adf_p < alpha      # reject H0 → stationary
    kpss_ok = kpss_p > alpha     # fail to reject H0 → stationary

    verdict = '✅ Dừng' if (adf_ok and kpss_ok) else '❌ Không dừng'

    print(f"\n  [{name}]")
    print(f"    ADF  stat={adf_stat:8.4f}  p={adf_p:.4e}  {'✅ Stationary' if adf_ok else '❌ Non-stationary'}")
    print(f"    KPSS stat={kpss_stat:8.4f}  p={kpss_p:.4e}  {'✅ Stationary' if kpss_ok else '❌ Non-stationary'}")
    print(f"    → Kết luận: {verdict}")
    return adf_ok and kpss_ok

is_stationary_raw = test_stationarity(y_train, 'revenue (raw)')

# Xác định d
if is_stationary_raw:
    d_order = 0
    y_diff  = y_train.copy()
    print(f"\n  d = 0 (chuỗi đã dừng)")
else:
    y_diff1 = np.diff(y_train, n=1)
    is_stationary_d1 = test_stationarity(y_diff1, 'revenue (diff=1)')
    if is_stationary_d1:
        d_order = 1
        y_diff  = y_diff1
        print(f"\n  d = 1 (sai phân bậc 1 đã dừng)")
    else:
        y_diff2 = np.diff(y_train, n=2)
        test_stationarity(y_diff2, 'revenue (diff=2)')
        d_order = 2
        y_diff  = y_diff2
        print(f"\n  d = 2 (sai phân bậc 2 đã dừng)")

print(f"\n  ✅ Chọn d = {d_order}")


### 2.3 Nhận Diện p, q (ACF/PACF)

In [ ]:

# ====================================================================
# BƯỚC 3 — NHẬN DIỆN p, q BẰNG ACF / PACF
# ====================================================================
print("\n" + "=" * 70)
print("BƯỚC 3 — NHẬN DIỆN p, q (ACF / PACF)")
print("=" * 70)

N_LAGS = min(60, len(y_diff) // 3)
acf_vals  = acf(y_diff,  nlags=N_LAGS, fft=True)
pacf_vals = pacf(y_diff, nlags=N_LAGS)

conf = 1.96 / np.sqrt(len(y_diff))

# Tìm lag đầu tiên nằm trong confidence band
def suggest_order(vals, conf, name):
    sig_lags = [i for i in range(1, len(vals)) if abs(vals[i]) > conf]
    if not sig_lags:
        return 0
    # Cutoff: lag lớn nhất liên tục từ 1
    order = 0
    for i in range(1, len(vals)):
        if abs(vals[i]) > conf:
            order = i
        else:
            break
    print(f"    {name}: significant lags = {sig_lags[:8]} → suggest order = {order}")
    return min(order, 3)   # cap tại 3 để tránh overfit

print("  Gợi ý từ correlogram:")
p_suggest = suggest_order(pacf_vals, conf, 'PACF (→ AR order p)')
q_suggest = suggest_order(acf_vals,  conf, 'ACF  (→ MA order q)')

# Seasonal: dữ liệu ngày → m=7 (tuần) hoặc 365 (năm)
# Với dữ liệu 10 năm, dùng m=7 (weekly seasonality) là hợp lý và feasible
M_SEASONAL = 7
print(f"\n  Seasonal period m = {M_SEASONAL} (weekly seasonality)")
print(f"  Gợi ý (p, d, q) = ({p_suggest}, {d_order}, {q_suggest})")



### 2.4 Lựa Chọn Mô Hình Bằng AIC (Grid-Search)

In [ ]:
# ====================================================================
# BƯỚC 4 — LỰA CHỌN MÔ HÌNH BẰNG AIC (Grid-search nhỏ)
# ====================================================================
print("\n" + "=" * 70)
print("BƯỚC 4 — LỰA CHỌN MÔ HÌNH BẰNG AIC")
print("=" * 70)
print("  Grid-search: p ∈ {0,1,2}, q ∈ {0,1,2}, d cố định, P∈{0,1}, Q∈{0,1}")
print("  (Dùng tập con đầu tiên để tăng tốc, sau đó train lại toàn bộ)\n")

# Dùng 20% đầu để grid-search nhanh
N_SUBSET  = min(len(y_train), 1000)
y_sub     = y_train[:N_SUBSET]
exog_sub  = exog_train[:N_SUBSET]

p_range = range(0, 3)
q_range = range(0, 3)
P_range = range(0, 2)
Q_range = range(0, 2)

aic_results = []

for p, q, P, Q in itertools.product(p_range, q_range, P_range, Q_range):
    # Bỏ qua (0,d,0)(0,0,0) — mô hình rỗng
    if p == 0 and q == 0 and P == 0 and Q == 0:
        continue
    try:
        res = SARIMAX(
            y_sub,
            exog=exog_sub,
            order=(p, d_order, q),
            seasonal_order=(P, 0, Q, M_SEASONAL),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False, maxiter=50)

        aic_results.append({
            'p': p, 'd': d_order, 'q': q,
            'P': P, 'D': 0, 'Q': Q, 'M': M_SEASONAL,
            'AIC': res.aic,
            'BIC': res.bic
        })
    except Exception:
        pass

aic_df = pd.DataFrame(aic_results).sort_values('AIC').reset_index(drop=True)

print(f"  {'p':>2} {'d':>2} {'q':>2}  {'P':>2} {'Q':>2}  {'AIC':>12}  {'BIC':>12}")
print("  " + "-" * 50)
for _, row in aic_df.head(8).iterrows():
    marker = " ← BEST" if _ == 0 else ""
    print(f"  {int(row.p):>2} {int(row.d):>2} {int(row.q):>2}  "
          f"{int(row.P):>2} {int(row.Q):>2}  "
          f"{row.AIC:>12.2f}  {row.BIC:>12.2f}{marker}")

best = aic_df.iloc[0]
BEST_ORDER   = (int(best.p), int(best.d), int(best.q))
BEST_SEASONAL = (int(best.P), 0, int(best.Q), M_SEASONAL)
print(f"\n  ✅ Best: SARIMAX{BEST_ORDER}×{BEST_SEASONAL}")



### 2.5 Huấn Luyện SARIMAX (toàn bộ train)

In [ ]:
# ====================================================================
# BƯỚC 5 — HUẤN LUYỆN SARIMAX (toàn bộ train)
# ====================================================================
print("\n" + "=" * 70)
print("BƯỚC 5 — HUẤN LUYỆN SARIMAX (toàn bộ train)")
print("=" * 70)

sarima_model = SARIMAX(
    y_train,
    exog=exog_train,
    order=BEST_ORDER,
    seasonal_order=BEST_SEASONAL,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False, maxiter=200)

print(sarima_fit.summary().tables[0].as_text())
print(sarima_fit.summary().tables[1].as_text())

y_train_pred = sarima_fit.fittedvalues

r2_tr  = r2_score(y_train, y_train_pred)
mae_tr = mean_absolute_error(y_train, y_train_pred)
rmse_tr = np.sqrt(mean_squared_error(y_train, y_train_pred))

print(f"\n  Train R²   : {r2_tr:.6f}")
print(f"  Train MAE  : {mae_tr:>12,.2f}")
print(f"  Train RMSE : {rmse_tr:>12,.2f}")

# ====================================================================


### 2.6 Chuẩn Đoán Phần Dư

In [ ]:
# BƯỚC 6 — CHUẨN ĐOÁN PHẦN DƯ
# ====================================================================
print("\n" + "=" * 70)
print("BƯỚC 6 — CHUẨN ĐOÁN PHẦN DƯ")
print("=" * 70)

residuals = sarima_fit.resid

# Ljung-Box test
lb_result = acorr_ljungbox(residuals, lags=[10, 20, 30], return_df=True)
print("\n  Ljung-Box test (H0: Phần dư là white noise):")
print(f"  {'Lag':>5}  {'LB stat':>10}  {'p-value':>10}  {'Kết quả'}")
print("  " + "-" * 45)
for lag, row in lb_result.iterrows():
    ok = '✅ White noise' if row['lb_pvalue'] > 0.05 else '❌ Còn tự tương quan'
    print(f"  {lag:>5}  {row['lb_stat']:>10.4f}  {row['lb_pvalue']:>10.4f}  {ok}")

# Durbin-Watson
dw = durbin_watson(residuals)
print(f"\n  Durbin-Watson: {dw:.4f}  ({'✅ Không tự tương quan' if 1.5 < dw < 2.5 else '⚠️  Có thể có tự tương quan'})")

# Shapiro-Wilk (normality)
if len(residuals) <= 5000:
    _, sw_p = stats.shapiro(residuals[:5000])
    print(f"  Shapiro-Wilk p={sw_p:.4e}  ({'✅ Normal' if sw_p > 0.05 else '⚠️  Non-normal (thường gặp với n lớn)'})")

# ====================================================================


### 2.7 Dự Báo & So Sánh với Ridge Baseline

In [ ]:
# BƯỚC 7 — DỰ BÁO TẬP KIỂM TRA & SO SÁNH
# ====================================================================
print("\n" + "=" * 70)
print("BƯỚC 7 — DỰ BÁO VÀ SO SÁNH")
print("=" * 70)

# SARIMAX forecast
if exog_test is not None and len(exog_test) > 0:
    forecast_res = sarima_fit.get_forecast(steps=len(y_test), exog=exog_test)
    y_pred_sarima  = forecast_res.predicted_mean
    conf_int       = forecast_res.conf_int(alpha=0.05)
    ci_lower = conf_int[:, 0]
    ci_upper = conf_int[:, 1]

    r2_sarima   = r2_score(y_test, y_pred_sarima)
    mae_sarima  = mean_absolute_error(y_test, y_pred_sarima)
    rmse_sarima = np.sqrt(mean_squared_error(y_test, y_pred_sarima))
    mape_sarima = np.mean(np.abs((y_test - y_pred_sarima) / np.where(y_test == 0, 1, y_test))) * 100
else:
    # Nếu không có test data thực tế → tạo out-of-sample forecast
    TEST_STEPS = 547  # ~18 tháng
    test_dates_fc = pd.date_range(TEST_START, periods=TEST_STEPS, freq='D')
    test_months   = test_dates_fc.month
    exog_fc = np.column_stack([
        np.full(TEST_STEPS, df_train['shipping_fee'].median()),
        np.full(TEST_STEPS, df_train['num_returns_orders'].median()
                if 'num_returns_orders' in df_train.columns else 0),
        (test_months.isin([4,5,6])).astype(int)
    ])
    forecast_res  = sarima_fit.get_forecast(steps=TEST_STEPS, exog=exog_fc[:, :len(EXOG_COLS)])
    y_pred_sarima = forecast_res.predicted_mean
    conf_int      = forecast_res.conf_int(alpha=0.05)
    ci_lower      = conf_int[:, 0]
    ci_upper      = conf_int[:, 1]
    y_test        = None
    r2_sarima = mae_sarima = rmse_sarima = mape_sarima = None

# Ridge baseline (cho so sánh)
scaler_r = StandardScaler()
X_tr_sc  = scaler_r.fit_transform(exog_train)
ridge_bl  = Ridge(alpha=10.0).fit(X_tr_sc, y_train)

if y_test is not None:
    X_te_sc     = scaler_r.transform(exog_test)
    y_pred_ridge = ridge_bl.predict(X_te_sc)
    r2_ridge    = r2_score(y_test, y_pred_ridge)
    mae_ridge   = mean_absolute_error(y_test, y_pred_ridge)
    rmse_ridge  = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
    mape_ridge  = np.mean(np.abs((y_test - y_pred_ridge) / np.where(y_test==0,1,y_test))) * 100

    print(f"\n  {'Metric':<10} {'SARIMAX':>14} {'Ridge baseline':>16}")
    print("  " + "-" * 42)
    print(f"  {'R²':<10} {r2_sarima:>14.4f} {r2_ridge:>16.4f}")
    print(f"  {'MAE':<10} {mae_sarima:>14,.0f} {mae_ridge:>16,.0f}")
    print(f"  {'RMSE':<10} {rmse_sarima:>14,.0f} {rmse_ridge:>16,.0f}")
    print(f"  {'MAPE%':<10} {mape_sarima:>14.2f} {mape_ridge:>16.2f}")

    winner = 'SARIMAX' if r2_sarima > r2_ridge else 'Ridge'
    print(f"\n  🏆 Winner (R²): {winner}")
else:
    print("  (Không có test label → hiển thị out-of-sample forecast)")



### 2.8 Visualization Tổng Hợp

In [ ]:
# ====================================================================
# BƯỚC 8 — VISUALIZATION TỔNG HỢP
# ====================================================================
print("\n" + "=" * 70)
print("BƯỚC 8 — VISUALIZATION")
print("=" * 70)

fig = plt.figure(figsize=(22, 20))
fig.patch.set_facecolor(C_BG)
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.52, wspace=0.35)

# ── 1. Chuỗi gốc + diff ───────────────────────────────────────────
ax = fig.add_subplot(gs[0, :2])
style(ax, 'Revenue gốc vs Sai phân bậc d', 'Ngày', 'Revenue')
ax2_twin = ax.twinx()
ax.plot(df_train['date'], y_train, color=C_NEON, linewidth=0.6, alpha=0.8, label='Revenue gốc')
if d_order > 0:
    diff_dates = df_train['date'].values[d_order:]
    ax2_twin.plot(diff_dates, y_diff, color=C_GOLD, linewidth=0.5, alpha=0.6, label=f'Diff={d_order}')
    ax2_twin.set_ylabel(f'Sai phân (d={d_order})', color=C_GOLD, fontsize=8)
    ax2_twin.tick_params(colors=C_GOLD)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8,
          facecolor=C_PANEL, labelcolor='white', loc='upper left')
ax.xaxis.set_major_locator(plt.MaxNLocator(8))

# ── 2. ADF/KPSS summary box ───────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
ax.set_facecolor(C_PANEL)
ax.axis('off')
info_text = (
    f"STATIONARITY RESULTS\n"
    f"{'─'*28}\n"
    f"Raw series:\n"
    f"  d = {d_order} → {'Dừng' if d_order==0 else f'Cần diff({d_order})'}\n\n"
    f"BEST MODEL (AIC)\n"
    f"{'─'*28}\n"
    f"SARIMAX({BEST_ORDER[0]},{BEST_ORDER[1]},{BEST_ORDER[2]})\n"
    f"      ×({BEST_SEASONAL[0]},0,{BEST_SEASONAL[2]},{M_SEASONAL})\n"
    f"AIC = {aic_df.iloc[0]['AIC']:.2f}\n\n"
    f"EXOGENOUS:\n"
    + "\n".join(f"  • {c}" for c in EXOG_COLS)
)
ax.text(0.05, 0.95, info_text, transform=ax.transAxes,
        color=C_NEON, fontsize=9, va='top', family='monospace',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#1a1a35', edgecolor=C_NEON, alpha=0.8))
ax.set_title('Model Summary', color=C_NEON, fontsize=10, fontweight='bold')

# ── 3. ACF ────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
plot_acf(y_diff, lags=40, ax=ax, color=C_NEON, alpha=0.6)
style(ax, f'ACF (chuỗi diff={d_order})', 'Lag', 'ACF')
ax.axhline(conf,  color=C_RED,  linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(-conf, color=C_RED,  linestyle='--', linewidth=1, alpha=0.7)

# ── 4. PACF ───────────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
plot_pacf(y_diff, lags=40, ax=ax, color=C_GOLD, alpha=0.6, method='ywm')
style(ax, f'PACF (chuỗi diff={d_order})', 'Lag', 'PACF')
ax.axhline(conf,  color=C_RED, linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(-conf, color=C_RED, linestyle='--', linewidth=1, alpha=0.7)

# ── 5. AIC heatmap (p vs q, P=best.P, Q=best.Q) ──────────────────
ax = fig.add_subplot(gs[1, 2])
style(ax, 'AIC Grid (p vs q)', 'q', 'p')
aic_sub = aic_df[(aic_df.P == best.P) & (aic_df.Q == best.Q)].copy()
if len(aic_sub) >= 2:
    pivot = aic_sub.pivot_table(index='p', columns='q', values='AIC')
    im = ax.imshow(pivot.values, cmap='plasma', aspect='auto')
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns.astype(int))
    ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index.astype(int))
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                        fontsize=7, color='white', fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(colors='white')
    ax.set_title(f'AIC Grid (P={int(best.P)}, Q={int(best.Q)})', color=C_NEON, fontsize=10, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Grid quá nhỏ\nđể hiển thị', ha='center', va='center',
            color=C_NEON, fontsize=10, transform=ax.transAxes)

# ── 6. Fitted vs Actual (train) ───────────────────────────────────
ax = fig.add_subplot(gs[2, :2])
style(ax, 'SARIMAX — Fitted vs Actual (Train)', 'Ngày', 'Revenue')
ax.plot(df_train['date'], y_train,      color=C_GRAY, linewidth=0.6, alpha=0.7, label='Actual')
ax.plot(df_train['date'], y_train_pred, color=C_NEON, linewidth=0.8, alpha=0.9, label='Fitted')
ax.legend(fontsize=8, facecolor=C_PANEL, labelcolor='white')
ax.set_title(f'SARIMAX Fitted vs Actual (Train R²={r2_tr:.4f})',
             color=C_NEON, fontsize=10, fontweight='bold', pad=8)
ax.xaxis.set_major_locator(plt.MaxNLocator(8))

# ── 7. Residuals ──────────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 2])
style(ax, 'Phần dư theo thời gian', 'Ngày', 'Residual')
ax.plot(df_train['date'], residuals, color=C_GOLD, linewidth=0.4, alpha=0.7)
ax.axhline(0, color=C_RED, linestyle='--', linewidth=1)
ax.fill_between(df_train['date'], residuals, 0,
                where=(residuals > 0), color=C_GREEN, alpha=0.15)
ax.fill_between(df_train['date'], residuals, 0,
                where=(residuals < 0), color=C_RED, alpha=0.15)
ax.xaxis.set_major_locator(plt.MaxNLocator(6))

# ── 8. Q-Q Plot ───────────────────────────────────────────────────
ax = fig.add_subplot(gs[3, 0])
style(ax, 'Q-Q Plot (Phần dư)', 'Theoretical Quantiles', 'Sample Quantiles')
(osm, osr), (slope, intercept, r) = stats.probplot(residuals)
ax.scatter(osm, osr, color=C_NEON, s=4, alpha=0.4)
ax.plot(osm, slope * np.array(osm) + intercept, color=C_RED, linewidth=1.5)

# ── 9. Residual histogram ─────────────────────────────────────────
ax = fig.add_subplot(gs[3, 1])
style(ax, 'Phân phối phần dư', 'Residual', 'Frequency')
ax.hist(residuals, bins=60, color=C_NEON, alpha=0.6, edgecolor='none')
xr = np.linspace(residuals.min(), residuals.max(), 200)
pdf = stats.norm.pdf(xr, residuals.mean(), residuals.std()) * len(residuals) * (residuals.max()-residuals.min()) / 60
ax.plot(xr, pdf, color=C_RED, linewidth=2, label='Normal fit')
ax.legend(fontsize=8, facecolor=C_PANEL, labelcolor='white')

# ── 10. Forecast vs Actual (test) ────────────────────────────────
ax = fig.add_subplot(gs[3, 2])
style(ax, 'Dự báo tập kiểm tra', 'Ngày', 'Revenue')
fc_dates = df_test['date'].values if len(df_test) > 0 else pd.date_range(TEST_START, periods=len(y_pred_sarima), freq='D')
ax.fill_between(fc_dates, ci_lower, ci_upper, color=C_NEON, alpha=0.12, label='95% CI')
ax.plot(fc_dates, y_pred_sarima, color=C_NEON, linewidth=1.2, label='SARIMAX forecast')
if y_test is not None:
    ax.plot(fc_dates, y_test, color=C_GRAY, linewidth=0.8, alpha=0.7, label='Actual')
    ax.plot(fc_dates, y_pred_ridge, color=C_GOLD, linewidth=0.8, linestyle='--', alpha=0.8, label='Ridge baseline')
ax.legend(fontsize=7, facecolor=C_PANEL, labelcolor='white')
ax.xaxis.set_major_locator(plt.MaxNLocator(5))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig.suptitle(
    'SARIMAX PIPELINE — Revenue Forecasting\n'
    f'Best: SARIMAX{BEST_ORDER}×{BEST_SEASONAL}  |  Exog: {EXOG_COLS}',
    color=C_NEON, fontsize=14, fontweight='bold', y=1.01
)
plt.savefig('sarima_pipeline.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()
print("💾 sarima_pipeline.png đã lưu")

# ── Tóm tắt cuối ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("TỔNG KẾT SARIMA PIPELINE")
print("=" * 70)
print(f"""
  Mô hình   : SARIMAX{BEST_ORDER}×{BEST_SEASONAL}
  d (diff)  : {d_order}  (chọn qua ADF + KPSS)
  p, q      : {BEST_ORDER[0]}, {BEST_ORDER[2]}  (chọn qua ACF/PACF + AIC grid)
  AIC best  : {aic_df.iloc[0]['AIC']:.2f}
  Exogenous : {EXOG_COLS}
  Train R²  : {r2_tr:.4f}
""")
if y_test is not None:
    print(f"  Test SARIMAX → R²={r2_sarima:.4f}  MAE={mae_sarima:,.0f}  MAPE={mape_sarima:.2f}%")
    print(f"  Test Ridge   → R²={r2_ridge:.4f}  MAE={mae_ridge:,.0f}  MAPE={mape_ridge:.2f}%")
    print(f"  🏆 Winner    : {winner}")
print("=" * 70)
print("✅ SARIMA PIPELINE HOÀN THÀNH!")
print("=" * 70)

---
# 🤖 FILE 3 — Deep Learning & Ensemble Pipeline


### 3.1 Import & Kiểm Tra Thư Viện

In [ ]:
"""
====================================================================
FILE 2 — DEEP LEARNING & ENSEMBLE PIPELINE
====================================================================
So sánh 4 mô hình:
  1.  XGBoost          (Gradient Boosting, nhanh, interpretable)
  2.  Random Forest    (Bagging ensemble, robust)
  3.  LSTM             (Deep Learning, sequential memory)
  4.  Temporal Fusion  (Transformer-based, attention mechanism)

Các bước:
  A.  ETL & Feature Engineering
  B.  TimeSeriesSplit CV cho mỗi mô hình
  C.  Huấn luyện & đánh giá
  D.  SHAP cho XGBoost (giải thích mô hình)
  E.  So sánh tổng hợp → chọn winner
  F.  Dự báo tập kiểm tra với mô hình tốt nhất
  G.  Visualization đầy đủ
====================================================================
Cài đặt (nếu chưa có):
  pip install xgboost shap torch
====================================================================
"""

import warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble        import RandomForestRegressor

# ─── Kiểm tra thư viện optional ───────────────────────────────────
try:
    import xgboost as xgb
    XGB_OK = True
except ImportError:
    XGB_OK = False
    print("⚠️  pip install xgboost")

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False
    print("⚠️  pip install shap")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"✅ PyTorch device: {DEVICE}")
except ImportError:
    TORCH_OK = False
    print("⚠️  pip install torch — LSTM & Transformer sẽ bị bỏ qua")

# ─── Màu sắc ──────────────────────────────────────────────────────
C_BG    = '#0d0d1a'
C_PANEL = '#13132b'
C_NEON  = '#00e5ff'
C_GOLD  = '#ffd700'
C_RED   = '#ff4560'
C_GREEN = '#00e096'
C_PURPLE= '#bf5fff'
C_ORG   = '#ff9800'
C_GRAY  = '#555577'

MODEL_COLORS = {
    'XGBoost':         C_NEON,
    'RandomForest':    C_GREEN,
    'LSTM':            C_GOLD,
    'TFT':             C_PURPLE,
}

plt.rcParams.update({
    'figure.facecolor': C_BG, 'axes.facecolor': C_PANEL,
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#333355',
    'grid.color': '#1e1e3f', 'grid.linestyle': '--', 'grid.alpha': 0.5,
})

def style(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(C_PANEL)
    for s in ['top','right']:
        ax.spines[s].set_visible(False)
    for s in ['bottom','left']:
        ax.spines[s].set_color('#333355')
    if title:  ax.set_title(title,  color=C_NEON, fontsize=10, fontweight='bold', pad=8)
    if xlabel: ax.set_xlabel(xlabel, color='#aaaacc', fontsize=8)
    if ylabel: ax.set_ylabel(ylabel, color='#aaaacc', fontsize=8)
    ax.grid(True)



### 3.2 ETL & Feature Engineering

In [ ]:
# ====================================================================
# A. ETL & FEATURE ENGINEERING
# ====================================================================
print("=" * 70)
print("A — ETL & FEATURE ENGINEERING")
print("=" * 70)

sales       = pd.read_csv('sales.csv',       parse_dates=['Date'])
orders      = pd.read_csv('orders.csv',      parse_dates=['order_date'])
order_items = pd.read_csv('order_items.csv')
payments    = pd.read_csv('payments.csv')
shipments   = pd.read_csv('shipments.csv',   parse_dates=['ship_date','delivery_date'])
returns_df  = pd.read_csv('returns.csv',     parse_dates=['return_date'])

sales = sales.rename(columns={'Date':'date','Revenue':'revenue','COGS':'cogs'})
sales['date'] = sales['date'].dt.normalize()

shipments['delivery_days'] = (shipments['delivery_date'] - shipments['ship_date']).dt.days
shipments_slim = shipments[['order_id','shipping_fee','delivery_days']]

order_items_agg = order_items.groupby('order_id', as_index=False).agg(
    discount_amount=('discount_amount','sum'),
    quantity=('quantity','sum'),
    has_promotion=('promo_id', lambda x: int(x.notna().any()))
)
payments_slim = payments[['order_id','payment_method']]

orders_full = (
    orders[['order_id','order_date']]
    .merge(shipments_slim,  on='order_id', how='left')
    .merge(order_items_agg, on='order_id', how='left')
    .merge(payments_slim,   on='order_id', how='left')
)
for col in ['shipping_fee','delivery_days','discount_amount','quantity','has_promotion']:
    orders_full[col] = orders_full[col].fillna(0)
orders_full['order_date'] = orders_full['order_date'].dt.normalize()

daily_orders = orders_full.groupby('order_date', as_index=False).agg(
    shipping_fee    =('shipping_fee',   'sum'),
    discount_amount =('discount_amount','sum'),
    quantity        =('quantity',       'sum'),
    promo_orders    =('has_promotion',  'sum'),
).rename(columns={'order_date':'date'})

daily_returns = returns_df.groupby('return_date', as_index=False).agg(
    num_returns_orders=('return_id','count'),
    refund_amount=('refund_amount','sum'),
).rename(columns={'return_date':'date'})
daily_returns['date'] = daily_returns['date'].dt.normalize()

df = sales.copy()
for tbl in [daily_orders, daily_returns]:
    df = df.merge(tbl, on='date', how='left')

for col in ['shipping_fee','discount_amount','quantity','promo_orders',
            'num_returns_orders','refund_amount']:
    if col in df.columns:
        df[col] = df[col].fillna(0)

df = df.sort_values('date').reset_index(drop=True)
df['date'] = pd.to_datetime(df['date'])

# Feature engineering
df['month']       = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['year']        = df['date'].dt.year
df['is_peak_season'] = df['month'].isin([4,5,6]).astype(int)
df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Lag + rolling (shift(1) để tránh leakage)
for lag in [1, 7, 14, 30]:
    df[f'rev_lag_{lag}'] = df['revenue'].shift(lag)
for win in [7, 14, 30]:
    df[f'rev_roll_{win}'] = df['revenue'].shift(1).rolling(win, min_periods=1).mean()
    df[f'rev_std_{win}']  = df['revenue'].shift(1).rolling(win, min_periods=1).std()

df['target'] = df['revenue']

FEATURE_COLS = [
    # 3 biến chính
    'shipping_fee', 'num_returns_orders', 'is_peak_season',
    # Temporal
    'is_weekend', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
    # Lags
    'rev_lag_1', 'rev_lag_7', 'rev_lag_14', 'rev_lag_30',
    # Rolling
    'rev_roll_7', 'rev_roll_14', 'rev_roll_30',
    'rev_std_7',  'rev_std_30',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

TRAIN_END  = '2022-12-31'
TEST_START = '2023-01-01'
TEST_END   = '2024-07-01'

df_all   = df[['date','target'] + FEATURE_COLS].dropna().copy().reset_index(drop=True)
df_train = df_all[df_all['date'] <= TRAIN_END].copy().reset_index(drop=True)
df_test  = df_all[(df_all['date'] >= TEST_START) & (df_all['date'] <= TEST_END)].copy().reset_index(drop=True)

X_train = df_train[FEATURE_COLS].values
y_train = df_train['target'].values
X_test  = df_test[FEATURE_COLS].values  if len(df_test) > 0 else None
y_test  = df_test['target'].values      if len(df_test) > 0 else None

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test) if X_test is not None else None

print(f"✅ Train: {df_train['date'].min().date()} → {df_train['date'].max().date()} ({len(df_train):,})")
print(f"   Test : {df_test['date'].min().date() if len(df_test)>0 else 'N/A'} "
      f"→ {df_test['date'].max().date() if len(df_test)>0 else 'N/A'} ({len(df_test):,})")
print(f"   Features: {len(FEATURE_COLS)}")



### 3.3 XGBoost

In [ ]:
# ====================================================================
# B. HÀM ĐÁNH GIÁ & CV HELPER
# ====================================================================
N_SPLITS = 5
tscv     = TimeSeriesSplit(n_splits=N_SPLITS)

def eval_metrics(y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true==0,1,y_true))) * 100
    return {'R²': r2, 'MAE': mae, 'RMSE': rmse, 'MAPE%': mape}

results      = {}   # {model_name: {cv_r2_mean, cv_r2_std, test_metrics, y_pred_test}}
cv_fold_r2   = {}   # {model_name: [r2_fold1, ...]}

# ====================================================================
# C1. XGBOOST
# ====================================================================
print("\n" + "=" * 70)
print("C1 — XGBOOST")
print("=" * 70)

if XGB_OK:
    xgb_params = dict(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1,
        early_stopping_rounds=30, eval_metric='rmse'
    )

    fold_r2_xgb = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train_sc), 1):
        Xtr, Xvl = X_train_sc[tr_idx], X_train_sc[val_idx]
        ytr, yvl = y_train[tr_idx],    y_train[val_idx]
        clf = xgb.XGBRegressor(**xgb_params)
        clf.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], verbose=False)
        r2_fold = r2_score(yvl, clf.predict(Xvl))
        fold_r2_xgb.append(r2_fold)
        print(f"  Fold {fold}: R²={r2_fold:+.4f}")

    # Train cuối
    xgb_final = xgb.XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1
    )
    xgb_final.fit(X_train_sc, y_train)
    y_pred_xgb_test = xgb_final.predict(X_test_sc) if X_test_sc is not None else None

    cv_fold_r2['XGBoost'] = fold_r2_xgb
    results['XGBoost'] = {
        'cv_r2_mean': np.mean(fold_r2_xgb),
        'cv_r2_std':  np.std(fold_r2_xgb),
        'test_metrics': eval_metrics(y_test, y_pred_xgb_test) if y_test is not None else {},
        'y_pred_test': y_pred_xgb_test,
        'model': xgb_final
    }
    print(f"  CV R² mean = {np.mean(fold_r2_xgb):+.4f} ± {np.std(fold_r2_xgb):.4f}")
else:
    print("  ⚠️  XGBoost không khả dụng")



### 3.4 Random Forest

In [ ]:
# ====================================================================
# C2. RANDOM FOREST
# ====================================================================
print("\n" + "=" * 70)
print("C2 — RANDOM FOREST")
print("=" * 70)

rf_params = dict(
    n_estimators=300, max_depth=12, min_samples_leaf=5,
    max_features=0.7, random_state=42, n_jobs=-1
)

fold_r2_rf = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train_sc), 1):
    Xtr, Xvl = X_train_sc[tr_idx], X_train_sc[val_idx]
    ytr, yvl = y_train[tr_idx],    y_train[val_idx]
    clf = RandomForestRegressor(**rf_params)
    clf.fit(Xtr, ytr)
    r2_fold = r2_score(yvl, clf.predict(Xvl))
    fold_r2_rf.append(r2_fold)
    print(f"  Fold {fold}: R²={r2_fold:+.4f}")

rf_final = RandomForestRegressor(**rf_params)
rf_final.fit(X_train_sc, y_train)
y_pred_rf_test = rf_final.predict(X_test_sc) if X_test_sc is not None else None

cv_fold_r2['RandomForest'] = fold_r2_rf
results['RandomForest'] = {
    'cv_r2_mean': np.mean(fold_r2_rf),
    'cv_r2_std':  np.std(fold_r2_rf),
    'test_metrics': eval_metrics(y_test, y_pred_rf_test) if y_test is not None else {},
    'y_pred_test': y_pred_rf_test,
    'model': rf_final
}
print(f"  CV R² mean = {np.mean(fold_r2_rf):+.4f} ± {np.std(fold_r2_rf):.4f}")



### 3.5 LSTM

In [ ]:
# ====================================================================
# C3. LSTM
# ====================================================================
print("\n" + "=" * 70)
print("C3 — LSTM")
print("=" * 70)

SEQ_LEN = 30   # lookback window

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

if TORCH_OK:
    class LSTMModel(nn.Module):
        def __init__(self, input_size, hidden=128, num_layers=2, dropout=0.2):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hidden, num_layers=num_layers,
                                batch_first=True, dropout=dropout)
            self.fc   = nn.Sequential(
                nn.Linear(hidden, 64),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(64, 1)
            )
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :]).squeeze(-1)

    def train_lstm(X_seq, y_seq, epochs=50, lr=1e-3, batch=64):
        ds      = TensorDataset(torch.from_numpy(X_seq), torch.from_numpy(y_seq))
        loader  = DataLoader(ds, batch_size=batch, shuffle=False)
        model   = LSTMModel(X_seq.shape[2]).to(DEVICE)
        opt     = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        sched   = torch.optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5)
        loss_fn = nn.HuberLoss()
        model.train()
        for ep in range(epochs):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
        return model

    def predict_lstm(model, X_seq):
        model.eval()
        with torch.no_grad():
            t = torch.from_numpy(X_seq.astype(np.float32)).to(DEVICE)
            return model(t).cpu().numpy()

    # Chuẩn hoá y cho LSTM
    y_scaler   = StandardScaler()
    y_train_sc = y_scaler.fit_transform(y_train.reshape(-1,1)).ravel().astype(np.float32)

    X_seq_full, y_seq_full = make_sequences(X_train_sc.astype(np.float32), y_train_sc, SEQ_LEN)

    fold_r2_lstm = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_seq_full), 1):
        Xtr, Xvl = X_seq_full[tr_idx], X_seq_full[val_idx]
        ytr, yvl = y_seq_full[tr_idx],  y_seq_full[val_idx]
        m = train_lstm(Xtr, ytr, epochs=40, lr=5e-4)
        pred_sc  = predict_lstm(m, Xvl)
        pred_rev = y_scaler.inverse_transform(pred_sc.reshape(-1,1)).ravel()
        true_rev = y_scaler.inverse_transform(yvl.reshape(-1,1)).ravel()
        r2_fold  = r2_score(true_rev, pred_rev)
        fold_r2_lstm.append(r2_fold)
        print(f"  Fold {fold}: R²={r2_fold:+.4f}")

    # Final LSTM
    lstm_final = train_lstm(X_seq_full, y_seq_full, epochs=60, lr=5e-4)

    if X_test_sc is not None and len(X_test_sc) >= SEQ_LEN:
        X_all_sc  = np.vstack([X_train_sc, X_test_sc]).astype(np.float32)
        y_all_sc  = y_scaler.transform(
            np.concatenate([y_train, y_test]).reshape(-1,1)).ravel().astype(np.float32)
        X_seq_all, y_seq_all = make_sequences(X_all_sc, y_all_sc, SEQ_LEN)
        n_test_seq = len(y_test) - SEQ_LEN if len(y_test) > SEQ_LEN else len(y_test)
        pred_sc    = predict_lstm(lstm_final, X_seq_all[-len(y_test):])
        y_pred_lstm_test = y_scaler.inverse_transform(pred_sc.reshape(-1,1)).ravel()
        y_test_aligned   = y_test[-len(y_pred_lstm_test):]
    else:
        y_pred_lstm_test = None
        y_test_aligned   = y_test

    cv_fold_r2['LSTM'] = fold_r2_lstm
    results['LSTM'] = {
        'cv_r2_mean': np.mean(fold_r2_lstm),
        'cv_r2_std':  np.std(fold_r2_lstm),
        'test_metrics': eval_metrics(y_test_aligned, y_pred_lstm_test)
                        if (y_pred_lstm_test is not None and y_test is not None) else {},
        'y_pred_test': y_pred_lstm_test,
        'model': lstm_final
    }
    print(f"  CV R² mean = {np.mean(fold_r2_lstm):+.4f} ± {np.std(fold_r2_lstm):.4f}")
else:
    print("  ⚠️  PyTorch không khả dụng — bỏ qua LSTM")



### 3.6 Temporal Fusion Transformer (TFT)

In [ ]:
# ====================================================================
# C4. TEMPORAL FUSION TRANSFORMER (TFT simplified)
# ====================================================================
print("\n" + "=" * 70)
print("C4 — TEMPORAL FUSION TRANSFORMER (Simplified)")
print("=" * 70)

if TORCH_OK:
    class TemporalFusionTransformer(nn.Module):
        """
        TFT đơn giản hoá:
          - Input projection
          - Multi-head Self-Attention (temporal dependencies)
          - Position-wise FFN
          - Gated Residual Network
          - Output head
        """
        def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, dropout=0.1):
            super().__init__()
            self.input_proj = nn.Linear(input_size, d_model)
            encoder_layer   = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead,
                dim_feedforward=d_model*4,
                dropout=dropout, batch_first=True,
                norm_first=True
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            # Gated Residual Network
            self.grn = nn.Sequential(
                nn.Linear(d_model, d_model),
                nn.ELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model, d_model),
                nn.Sigmoid()
            )
            self.out = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )

        def forward(self, x):
            x   = self.input_proj(x)              # (B, T, d_model)
            out = self.transformer(x)              # (B, T, d_model)
            gate = self.grn(out[:, -1, :])        # (B, d_model) — gate từ cuối chuỗi
            h    = out[:, -1, :] * gate
            return self.out(h).squeeze(-1)

    def train_tft(X_seq, y_seq, epochs=50, lr=5e-4, batch=64):
        ds      = TensorDataset(torch.from_numpy(X_seq), torch.from_numpy(y_seq))
        loader  = DataLoader(ds, batch_size=batch, shuffle=False)
        model   = TemporalFusionTransformer(X_seq.shape[2]).to(DEVICE)
        opt     = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
        sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        loss_fn = nn.HuberLoss()
        model.train()
        for ep in range(epochs):
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                loss_fn(model(xb), yb).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
        return model

    fold_r2_tft = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_seq_full), 1):
        Xtr, Xvl = X_seq_full[tr_idx], X_seq_full[val_idx]
        ytr, yvl = y_seq_full[tr_idx],  y_seq_full[val_idx]
        m = train_tft(Xtr, ytr, epochs=40)
        pred_sc  = predict_lstm(m, Xvl)   # reuse predict fn
        pred_rev = y_scaler.inverse_transform(pred_sc.reshape(-1,1)).ravel()
        true_rev = y_scaler.inverse_transform(yvl.reshape(-1,1)).ravel()
        r2_fold  = r2_score(true_rev, pred_rev)
        fold_r2_tft.append(r2_fold)
        print(f"  Fold {fold}: R²={r2_fold:+.4f}")

    tft_final = train_tft(X_seq_full, y_seq_full, epochs=60)

    if X_test_sc is not None and len(y_test) > 0:
        pred_sc_tft       = predict_lstm(tft_final, X_seq_all[-len(y_test):])
        y_pred_tft_test   = y_scaler.inverse_transform(pred_sc_tft.reshape(-1,1)).ravel()
    else:
        y_pred_tft_test = None

    cv_fold_r2['TFT'] = fold_r2_tft
    results['TFT'] = {
        'cv_r2_mean': np.mean(fold_r2_tft),
        'cv_r2_std':  np.std(fold_r2_tft),
        'test_metrics': eval_metrics(y_test_aligned, y_pred_tft_test)
                        if (y_pred_tft_test is not None and y_test is not None) else {},
        'y_pred_test': y_pred_tft_test,
        'model': tft_final
    }
    print(f"  CV R² mean = {np.mean(fold_r2_tft):+.4f} ± {np.std(fold_r2_tft):.4f}")
else:
    print("  ⚠️  PyTorch không khả dụng — bỏ qua TFT")



### 3.7 SHAP Explainer (XGBoost)

In [ ]:
# ====================================================================
# D. SHAP CHO XGBOOST
# ====================================================================
print("\n" + "=" * 70)
print("D — SHAP EXPLAINER (XGBoost)")
print("=" * 70)

shap_vals = None
shap_importance = None

if XGB_OK and SHAP_OK and 'XGBoost' in results:
    explainer    = shap.TreeExplainer(results['XGBoost']['model'])
    # Dùng tập con để tính SHAP nhanh hơn
    X_shap_sample = X_train_sc[:min(500, len(X_train_sc))]
    shap_vals     = explainer.shap_values(X_shap_sample)

    shap_importance = pd.DataFrame({
        'feature':       FEATURE_COLS,
        'mean_abs_shap': np.abs(shap_vals).mean(axis=0),
        'mean_shap':     shap_vals.mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

    print(f"\n  {'Rank':<5} {'Feature':<30} {'Mean |SHAP|':>12}  Direction")
    print("  " + "-" * 60)
    for i, row in shap_importance.head(10).iterrows():
        sign = "▲ +" if row['mean_shap'] > 0 else "▼ -"
        print(f"  {i+1:<5} {row['feature']:<30} {row['mean_abs_shap']:>12,.2f}  {sign}")

# ====================================================================
# E. SO SÁNH TỔNG HỢP → CHỌN WINNER


### 3.8 So Sánh Mô Hình → Chọn Winner

In [ ]:
# ====================================================================
print("\n" + "=" * 70)
print("E — SO SÁNH MÔ HÌNH")
print("=" * 70)

compare_rows = []
for name, res in results.items():
    row = {
        'Model':      name,
        'CV R² mean': res['cv_r2_mean'],
        'CV R² std':  res['cv_r2_std'],
    }
    if res['test_metrics']:
        row.update({f'Test {k}': v for k, v in res['test_metrics'].items()})
    compare_rows.append(row)

compare_df = pd.DataFrame(compare_rows).set_index('Model')
print("\n" + compare_df.to_string(float_format='{:,.4f}'.format))

# Chọn winner theo CV R²
best_model_name = max(results, key=lambda n: results[n]['cv_r2_mean'])
print(f"\n  🏆 Winner: {best_model_name}  "
      f"(CV R²={results[best_model_name]['cv_r2_mean']:+.4f})")

# ====================================================================
# F. DỰ BÁO TẬP KIỂM TRA VỚI WINNER
# ====================================================================
print("\n" + "=" * 70)
print(f"F — DỰ BÁO VỚI {best_model_name}")
print("=" * 70)

winner_pred = results[best_model_name]['y_pred_test']
if winner_pred is not None and y_test is not None:
    m = results[best_model_name]['test_metrics']
    print(f"  R²   : {m.get('R²',   'N/A')}")
    print(f"  MAE  : {m.get('MAE',  'N/A')}")
    print(f"  RMSE : {m.get('RMSE', 'N/A')}")
    print(f"  MAPE : {m.get('MAPE%','N/A')}")
else:
    print("  (Không có test labels để đánh giá)")

# ====================================================================
# G. VISUALIZATION
# ====================================================================
print("\n" + "=" * 70)


In [ ]:
# G. VISUALIZATION
# ====================================================================
print("\n" + "=" * 70)
print("G — VISUALIZATION")
print("=" * 70)

n_models  = len(results)
fig = plt.figure(figsize=(24, 22))
fig.patch.set_facecolor(C_BG)
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.52, wspace=0.35)

# ── 1. CV R² comparison ───────────────────────────────────────────
ax = fig.add_subplot(gs[0, :2])
style(ax, 'TimeSeriesSplit CV — R² per Fold per Model', 'Fold', 'R²')
x = np.arange(1, N_SPLITS + 1)
width = 0.2
offsets = np.linspace(-0.3, 0.3, len(cv_fold_r2))
for idx, (name, folds) in enumerate(cv_fold_r2.items()):
    col = MODEL_COLORS.get(name, C_GRAY)
    ax.bar(x + offsets[idx], folds, width=width, color=col,
           alpha=0.8, label=name, edgecolor='white', linewidth=0.3)
ax.axhline(0, color=C_RED, linestyle='--', linewidth=0.8, alpha=0.5)
ax.legend(fontsize=9, facecolor=C_PANEL, labelcolor='white', ncol=2)
ax.set_xticks(x)

# ── 2. CV R² mean ± std ───────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
style(ax, 'CV R² Mean ± Std', 'Model', 'R²')
names_list = list(results.keys())
means = [results[n]['cv_r2_mean'] for n in names_list]
stds  = [results[n]['cv_r2_std']  for n in names_list]
cols  = [MODEL_COLORS.get(n, C_GRAY) for n in names_list]
bars  = ax.bar(names_list, means, yerr=stds, color=cols,
               edgecolor='white', linewidth=0.5, capsize=4,
               error_kw=dict(ecolor='white', linewidth=1.5))
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(stds)*0.05,
            f'{val:+.3f}', ha='center', fontsize=8,
            color='white', fontweight='bold')
ax.axhline(0, color=C_RED, linestyle='--', linewidth=0.8)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=20, ha='right', fontsize=8)

# ── 3. SHAP importance ────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
style(ax, 'SHAP Feature Importance\n(XGBoost)', '|SHAP|', 'Feature')
if shap_importance is not None:
    top10 = shap_importance.head(10)
    bar_c = [C_GREEN if v > 0 else C_RED for v in top10['mean_shap']]
    ax.barh(top10['feature'], top10['mean_abs_shap'],
            color=bar_c, edgecolor='white', linewidth=0.3)
    ax.invert_yaxis()
    ax.tick_params(labelsize=7)
else:
    ax.text(0.5, 0.5, 'SHAP\nnot available', ha='center', va='center',
            color=C_NEON, fontsize=10, transform=ax.transAxes)

# ── 4. XGBoost feature importance (built-in) ──────────────────────
ax = fig.add_subplot(gs[1, 1])
style(ax, 'XGBoost Feature Importance\n(gain)', 'Gain', 'Feature')
if XGB_OK and 'XGBoost' in results:
    imp = results['XGBoost']['model'].feature_importances_
    imp_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': imp})\
               .sort_values('importance', ascending=False).head(10)
    ax.barh(imp_df['feature'], imp_df['importance'],
            color=C_NEON, edgecolor='white', linewidth=0.3)
    ax.invert_yaxis()
    ax.tick_params(labelsize=7)

# ── 5. RF feature importance ──────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
style(ax, 'Random Forest\nFeature Importance', 'Importance', 'Feature')
rf_imp = results['RandomForest']['model'].feature_importances_
rf_imp_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': rf_imp})\
              .sort_values('importance', ascending=False).head(10)
ax.barh(rf_imp_df['feature'], rf_imp_df['importance'],
        color=C_GREEN, edgecolor='white', linewidth=0.3)
ax.invert_yaxis()
ax.tick_params(labelsize=7)

# ── 6~9. Actual vs Predicted per model (test) ────────────────────
test_date_arr = df_test['date'].values if len(df_test) > 0 else None

model_names_plot = list(results.keys())
for plot_i, name in enumerate(model_names_plot[:4]):
    row_gs = 2 + plot_i // 2
    col_gs = plot_i % 2 + 1   # cols 1 & 2 → để col 0 cho residuals

    # Adjust layout: sử dụng 2×2 grid cho 4 model plots
    ax_pos = [(2, 0), (2, 1), (2, 2), (3, 0)]
    ax = fig.add_subplot(gs[ax_pos[plot_i][0], ax_pos[plot_i][1]])
    col = MODEL_COLORS.get(name, C_GRAY)
    style(ax, f'{name} — Test Forecast', 'Ngày', 'Revenue')

    pred = results[name]['y_pred_test']
    if pred is not None and test_date_arr is not None and y_test is not None:
        # Align lengths
        n = min(len(pred), len(y_test), len(test_date_arr))
        ax.plot(test_date_arr[-n:], y_test[-n:],  color=C_GRAY, linewidth=0.7, alpha=0.7, label='Actual')
        ax.plot(test_date_arr[-n:], pred[-n:],     color=col, linewidth=1.0, alpha=0.9, label=name)
        m = results[name]['test_metrics']
        ax.set_title(f"{name}\nR²={m.get('R²', 0):.3f}  MAE={m.get('MAE', 0):,.0f}",
                     color=col, fontsize=9, fontweight='bold', pad=6)
        ax.legend(fontsize=7, facecolor=C_PANEL, labelcolor='white')
    else:
        ax.text(0.5, 0.5, 'No test data', ha='center', va='center',
                color=col, fontsize=10, transform=ax.transAxes)
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=7)

# ── 10. Metrics heatmap ───────────────────────────────────────────
ax = fig.add_subplot(gs[3, 1:])
style(ax, 'Test Metrics Comparison', '', '')
ax.axis('off')

if any(results[n]['test_metrics'] for n in results):
    metric_names = ['R²', 'MAE', 'RMSE', 'MAPE%']
    heat_data, row_labels, col_labels = [], [], metric_names

    for name in results:
        m = results[name]['test_metrics']
        if m:
            row_labels.append(name)
            heat_data.append([m.get(k, np.nan) for k in metric_names])

    if heat_data:
        heat_arr = np.array(heat_data, dtype=float)
        # Normalize each column
        heat_norm = np.zeros_like(heat_arr)
        for j in range(heat_arr.shape[1]):
            col_j = heat_arr[:, j]
            col_j_valid = col_j[~np.isnan(col_j)]
            if len(col_j_valid) > 0 and col_j_valid.max() != col_j_valid.min():
                heat_norm[:, j] = (col_j - col_j_valid.min()) / (col_j_valid.max() - col_j_valid.min())

        im2 = ax.imshow(heat_norm, cmap='RdYlGn_r', aspect='auto',
                        vmin=0, vmax=1)
        ax.axis('on')
        ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, color='white', fontsize=9)
        ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, color='white', fontsize=9)

        for i in range(len(row_labels)):
            for j in range(len(col_labels)):
                val = heat_arr[i, j]
                if not np.isnan(val):
                    fmt = f'{val:.4f}' if j == 0 else f'{val:,.0f}' if j < 3 else f'{val:.2f}%'
                    ax.text(j, i, fmt, ha='center', va='center',
                            fontsize=9, color='white', fontweight='bold')
        ax.set_title('Test Metrics Heatmap (xanh = tốt hơn)',
                     color=C_NEON, fontsize=10, fontweight='bold', pad=8)
        plt.colorbar(im2, ax=ax, fraction=0.03, pad=0.02).ax.tick_params(colors='white')

# ── Winner badge ──────────────────────────────────────────────────
winner_r2 = results[best_model_name]['cv_r2_mean']
fig.text(0.5, 0.01,
         f"🏆  WINNER: {best_model_name}  |  CV R² = {winner_r2:+.4f}  |  "
         f"Dựa trên TimeSeriesSplit CV (n=5)",
         ha='center', color=C_GOLD, fontsize=12, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#1a1500', edgecolor=C_GOLD))

fig.suptitle(
    'REVENUE FORECASTING — MODEL COMPARISON\n'
    'XGBoost  ·  Random Forest  ·  LSTM  ·  Temporal Fusion Transformer',
    color=C_NEON, fontsize=14, fontweight='bold', y=1.01
)

plt.savefig('dl_model_comparison.png', dpi=150, bbox_inches='tight', facecolor=C_BG)
plt.show()
print("💾 dl_model_comparison.png đã lưu")



### 3.10 Tổng Kết & Submission

In [ ]:
# ====================================================================
# TỔNG KẾT
# ====================================================================
print("\n" + "=" * 70)
print("TỔNG KẾT SO SÁNH 4 MÔ HÌNH")
print("=" * 70)

print(f"\n  {'Model':<18} {'CV R² Mean':>12} {'CV R² Std':>12}"
      + (f"  {'Test R²':>10}  {'Test MAE':>12}" if any(r['test_metrics'] for r in results.values()) else ""))
print("  " + "-" * 65)

for name in results:
    res = results[name]
    row = f"  {name:<18} {res['cv_r2_mean']:>12.4f} {res['cv_r2_std']:>12.4f}"
    if res['test_metrics']:
        row += f"  {res['test_metrics'].get('R²',0):>10.4f}  {res['test_metrics'].get('MAE',0):>12,.0f}"
    print(row)

print(f"""
  🏆 Winner   : {best_model_name}
  📊 Lý do    : CV R² cao nhất trên TimeSeriesSplit (đúng chiều thời gian)
  🔍 SHAP     : XGBoost có giải thích SHAP đầy đủ nhất
  ⚡ Tốc độ  : XGBoost/RF nhanh; LSTM/TFT cần GPU để scale

  Gợi ý sử dụng:
    • Production: XGBoost (nhanh, interpretable, SHAP)
    • Accuracy: {best_model_name} nếu latency không phải vấn đề
    • Explainability: XGBoost + SHAP waterfall plot
""")

print("=" * 70)
print("✅ DEEP LEARNING PIPELINE HOÀN THÀNH!")
print("=" * 70)
# H. TẠO FILE SUBMISSION
# ====================================================================
print("\n" + "=" * 70)
print("H — TẠO FILE SUBMISSION (2023-01-01 → 2024-07-01)")
print("=" * 70)

# ----------------------------------------------------------
# H1. Tạo date range cho test period
# ----------------------------------------------------------
test_dates = pd.date_range(start='2023-01-01', end='2024-07-01', freq='D')
df_submission_base = pd.DataFrame({'date': test_dates})

print(f"Số ngày cần dự báo: {len(df_submission_base)}")

# ----------------------------------------------------------
# H2. Tạo features cho test period
# Dùng toàn bộ df (train) để tính lag/rolling đúng cách
# ----------------------------------------------------------

# Gộp train dates + test dates vào 1 dataframe để tính lag liên tục
# Không để leakage: revenue của test period = NaN ban đầu
df_full = df[['date', 'revenue'] + [c for c in FEATURE_COLS
              if c not in ['rev_lag_1','rev_lag_7','rev_lag_14','rev_lag_30',
                           'rev_roll_7','rev_roll_14','rev_roll_30',
                           'rev_std_7','rev_std_30']]].copy()

# Thêm các ngày test với revenue = NaN
df_test_template = pd.DataFrame({'date': test_dates})
df_test_template['revenue'] = np.nan

# Merge features có sẵn cho test dates (orders, returns, web)
# Các ngày 2023-2024 sẽ không có transaction data thực
# → Dùng giá trị trung bình tháng tương ứng từ train làm proxy
for col in ['shipping_fee', 'num_returns_orders', 'promo_orders',
            'discount_amount', 'quantity', 'refund_amount']:
    if col not in df_test_template.columns and col in df.columns:
        # Tính mean theo tháng từ train
        monthly_mean = (
            df[['date', col]].copy()
            .assign(month=lambda x: x['date'].dt.month)
            .groupby('month')[col]
            .mean()
        )
        df_test_template['month'] = df_test_template['date'].dt.month
        df_test_template[col] = df_test_template['month'].map(monthly_mean)
        df_test_template = df_test_template.drop(columns=['month'])

# Tạo temporal features cho test
df_test_template['month']          = df_test_template['date'].dt.month
df_test_template['day_of_week']    = df_test_template['date'].dt.dayofweek
df_test_template['year']           = df_test_template['date'].dt.year
df_test_template['is_peak_season'] = df_test_template['month'].isin([4,5,6]).astype(int)
df_test_template['is_weekend']     = (df_test_template['day_of_week'] >= 5).astype(int)
df_test_template['month_sin']      = np.sin(2 * np.pi * df_test_template['month'] / 12)
df_test_template['month_cos']      = np.cos(2 * np.pi * df_test_template['month'] / 12)
df_test_template['dow_sin']        = np.sin(2 * np.pi * df_test_template['day_of_week'] / 7)
df_test_template['dow_cos']        = np.cos(2 * np.pi * df_test_template['day_of_week'] / 7)

# ----------------------------------------------------------
# H3. Tính lag features đúng cách — không leakage
# Dùng revenue thực từ train, sau đó rolling predict
# ----------------------------------------------------------
print("Đang tính lag features cho test period...")

# Lấy revenue train (đã có thực)
revenue_history = df.set_index('date')['revenue'].to_dict()

# Iterative prediction để tính lag liên tục
predicted_revenue = {}

for i, row in df_test_template.iterrows():
    current_date = pd.Timestamp(row['date'])

    # Tính lag features từ revenue_history + predicted_revenue
    def get_revenue(date):
        """Lấy revenue từ history hoặc predicted"""
        if date in revenue_history:
            return revenue_history[date]
        elif date in predicted_revenue:
            return predicted_revenue[date]
        else:
            return np.nan

    # Lag values
    lag1  = get_revenue(current_date - pd.Timedelta(days=1))
    lag7  = get_revenue(current_date - pd.Timedelta(days=7))
    lag14 = get_revenue(current_date - pd.Timedelta(days=14))
    lag30 = get_revenue(current_date - pd.Timedelta(days=30))

    # Rolling mean (7 ngày trước)
    roll7_vals  = [get_revenue(current_date - pd.Timedelta(days=d))
                   for d in range(1, 8)]
    roll14_vals = [get_revenue(current_date - pd.Timedelta(days=d))
                   for d in range(1, 15)]
    roll30_vals = [get_revenue(current_date - pd.Timedelta(days=d))
                   for d in range(1, 31)]

    roll7_vals  = [v for v in roll7_vals  if not np.isnan(v)]
    roll14_vals = [v for v in roll14_vals if not np.isnan(v)]
    roll30_vals = [v for v in roll30_vals if not np.isnan(v)]

    df_test_template.at[i, 'rev_lag_1']   = lag1
    df_test_template.at[i, 'rev_lag_7']   = lag7
    df_test_template.at[i, 'rev_lag_14']  = lag14
    df_test_template.at[i, 'rev_lag_30']  = lag30
    df_test_template.at[i, 'rev_roll_7']  = np.mean(roll7_vals)  if roll7_vals  else np.nan
    df_test_template.at[i, 'rev_roll_14'] = np.mean(roll14_vals) if roll14_vals else np.nan
    df_test_template.at[i, 'rev_roll_30'] = np.mean(roll30_vals) if roll30_vals else np.nan
    df_test_template.at[i, 'rev_std_7']   = np.std(roll7_vals)   if len(roll7_vals)  > 1 else 0
    df_test_template.at[i, 'rev_std_30']  = np.std(roll30_vals)  if len(roll30_vals) > 1 else 0

    # Predict revenue cho ngày này bằng winner model
    feat_row = df_test_template.loc[i, FEATURE_COLS].values.reshape(1, -1)

    # Xử lý NaN trong features
    feat_row = np.nan_to_num(feat_row, nan=0.0)
    feat_row_sc = scaler.transform(feat_row)

    # Dùng winner model (XGBoost hoặc RF)
    if best_model_name in ['XGBoost', 'RandomForest']:
        pred_rev = results[best_model_name]['model'].predict(feat_row_sc)[0]
    else:
        # LSTM/TFT cần sequence → fallback về XGBoost nếu có
        if XGB_OK and 'XGBoost' in results:
            pred_rev = results['XGBoost']['model'].predict(feat_row_sc)[0]
        else:
            pred_rev = results['RandomForest']['model'].predict(feat_row_sc)[0]

    # Clip giá trị âm (revenue không thể âm)
    pred_rev = max(0, pred_rev)
    predicted_revenue[current_date] = pred_rev

    if i % 100 == 0:
        print(f"  Đã xử lý {i}/{len(df_test_template)} ngày...")

print(f"✅ Hoàn thành dự báo {len(predicted_revenue)} ngày")

# ----------------------------------------------------------
# H4. Tính COGS từ tỷ lệ COGS/Revenue của train
# ----------------------------------------------------------
# Tỷ lệ COGS trung bình từ train
cogs_ratio = (df['cogs'] / df['revenue'].replace(0, np.nan)).median()
print(f"COGS ratio (median từ train): {cogs_ratio:.4f}")

# ----------------------------------------------------------
# H5. Tạo submission DataFrame
# ----------------------------------------------------------
submission = pd.DataFrame({
    'Date':    [d.strftime('%Y-%m-%d') for d in test_dates],
    'Revenue': [round(predicted_revenue.get(pd.Timestamp(d), 0), 2)
                for d in test_dates],
    'COGS':    [round(predicted_revenue.get(pd.Timestamp(d), 0) * cogs_ratio, 2)
                for d in test_dates]
})

print(f"\nSubmission preview:")
print(submission.head(10).to_string(index=False))
print(f"\nShape: {submission.shape}")
print(f"Revenue range: {submission['Revenue'].min():,.2f} → {submission['Revenue'].max():,.2f}")
print(f"COGS range   : {submission['COGS'].min():,.2f} → {submission['COGS'].max():,.2f}")

# ----------------------------------------------------------
# H6. Lưu file
# ----------------------------------------------------------
submission.to_csv('submission.csv', index=False)
print("\n💾 Đã lưu: submission.csv")

# ----------------------------------------------------------
# H7. Visualize dự báo (không cần actual test)
# ----------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(20, 12))
fig.patch.set_facecolor(C_BG)

# Plot 1: Train actual + Test forecast
ax1 = axes[0]
ax1.set_facecolor(C_PANEL)

# Train revenue (sample cuối để dễ nhìn)
train_plot = df[df['date'] >= '2021-01-01'][['date','revenue']]
ax1.plot(train_plot['date'], train_plot['revenue'],
         color=C_GRAY, linewidth=0.8, alpha=0.7, label='Train Actual (2021–2022)')

# Test forecast
ax1.plot(submission['Date'], submission['Revenue'],
         color=MODEL_COLORS.get(best_model_name, C_NEON),
         linewidth=1.2, alpha=0.9,
         label=f'Forecast ({best_model_name})')

# Đường phân cách train/test
ax1.axvline('2023-01-01', color=C_RED, linewidth=1.5,
            linestyle='--', alpha=0.8, label='Train/Test split')

ax1.fill_between(submission['Date'], submission['Revenue'],
                 alpha=0.15,
                 color=MODEL_COLORS.get(best_model_name, C_NEON))

ax1.set_title(
    f'Revenue Forecast: 2023-01-01 → 2024-07-01  |  Model: {best_model_name}',
    color=C_NEON, fontsize=13, fontweight='bold', pad=10
)
ax1.set_xlabel('Date', color='#aaaacc')
ax1.set_ylabel('Revenue (VNĐ)', color='#aaaacc')
ax1.tick_params(colors='white', labelsize=8)
ax1.legend(fontsize=9, facecolor=C_PANEL, labelcolor='white')
ax1.grid(True, color='#1e1e3f', linestyle='--', alpha=0.5)
for spine in ['top','right']:
    ax1.spines[spine].set_visible(False)

# Plot 2: Monthly aggregated forecast
ax2 = axes[1]
ax2.set_facecolor(C_PANEL)

submission['date_dt'] = pd.to_datetime(submission['Date'])
monthly_forecast = (
    submission.groupby(submission['date_dt'].dt.to_period('M'))
    .agg({'Revenue': 'sum', 'COGS': 'sum'})
    .reset_index()
)
monthly_forecast['Profit'] = monthly_forecast['Revenue'] - monthly_forecast['COGS']
monthly_forecast['Period'] = monthly_forecast['date_dt'].astype(str)

x = np.arange(len(monthly_forecast))
w = 0.3
ax2.bar(x - w, monthly_forecast['Revenue'], width=w,
        color=C_NEON,   alpha=0.8, label='Revenue',  edgecolor='white', linewidth=0.3)
ax2.bar(x,     monthly_forecast['COGS'],    width=w,
        color=C_RED,    alpha=0.8, label='COGS',     edgecolor='white', linewidth=0.3)
ax2.bar(x + w, monthly_forecast['Profit'],  width=w,
        color=C_GREEN,  alpha=0.8, label='Profit',   edgecolor='white', linewidth=0.3)

ax2.set_xticks(x)
ax2.set_xticklabels(monthly_forecast['Period'],
                    rotation=45, ha='right', fontsize=8, color='white')
ax2.set_title('Monthly Revenue / COGS / Profit Forecast',
              color=C_NEON, fontsize=12, fontweight='bold', pad=10)
ax2.set_ylabel('VNĐ', color='#aaaacc')
ax2.tick_params(colors='white', labelsize=8)
ax2.legend(fontsize=9, facecolor=C_PANEL, labelcolor='white')
ax2.grid(True, color='#1e1e3f', linestyle='--', alpha=0.5)
for spine in ['top','right']:
    ax2.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig('submission_forecast.png', dpi=150,
            bbox_inches='tight', facecolor=C_BG)
plt.show()
print("💾 Đã lưu: submission_forecast.png")

# ----------------------------------------------------------
# H8. Kiểm tra format theo yêu cầu đề thi
# ----------------------------------------------------------
print("\n" + "=" * 50)
print("KIỂM TRA FORMAT SUBMISSION")
print("=" * 50)
print(f"Số cột    : {list(submission.columns)}")
print(f"Số dòng   : {len(submission)}")
print(f"Ngày đầu  : {submission['Date'].iloc[0]}")
print(f"Ngày cuối : {submission['Date'].iloc[-1]}")
print(f"Revenue < 0: {(submission['Revenue'] < 0).sum()} dòng")
print(f"COGS < 0   : {(submission['COGS'] < 0).sum()} dòng")
print(f"NaN        : {submission.isnull().sum().sum()} ô")

if submission['Revenue'].min() >= 0 and submission.isnull().sum().sum() == 0:
    print("\n✅ File hợp lệ — sẵn sàng nộp Kaggle")
else:
    print("\n⚠️  Cần kiểm tra lại trước khi nộp")